# Clase 11 — Carga Incremental: dos patrones sobre datos SAP
## Módulo 7 · SAP + Databricks Integration Real

**Caso de estudio:** Grupo Argos — actualización incremental de ventas y retorno de resultados ML a SAP

---

### Los dos patrones

| Patrón | Dirección | Tabla | Tecnología |
|--------|-----------|-------|------------|
| **MERGE Strategy** | SAP → Databricks Gold | `vbak_silver` → `gold_sales_kpis` | Delta MERGE + CASE WHEN + `updated_at` watermark |
| **CDF de Retorno** | Databricks Gold → SAP | `gold_customer_360` → OpenSharing | Change Data Feed |

### ¿Por qué BDC no habilita CDF hacia SAP Databricks?
> BDC garantiza que el Delta Share siempre tiene el snapshot más reciente de S/4HANA.
> La incrementalidad ocurre DENTRO de BDC via CDC interno (APRS).
> El receptor lee el snapshot actualizado — no necesita el historial de deltas.

### ¿Por qué la vuelta SÍ usa CDF?
> Cuando Databricks publica resultados ML de vuelta a SAP Datasphere,
> solo deben viajar los registros que cambiaron — no toda la tabla.
> CDF con `enableDeletionVectors=false` garantiza compatibilidad con BDC.

---
**Workspace:** Free Edition · `laboratory_sap_dev.sap_course`
**Verificación desde el receptor:** Trial ecotickets · `sap_gold_shared`


## 0. Setup

In [0]:
CATALOG = 'laboratory_sap_dev'
SCHEMA  = 'sap_course'
spark.sql(f'USE CATALOG {CATALOG}')
spark.sql(f'USE SCHEMA {SCHEMA}')
print(f'OK: {CATALOG}.{SCHEMA}')

for t in ['vbak_silver','gold_sales_kpis','gold_customer_360']:
    try:
        print(f'OK {t}: {spark.table(t).count():,} filas')
    except Exception as e:
        print(f'ERR {t}: {e}')


---
## Ejercicio 1 — MERGE Strategy: nuevas órdenes SAP → Gold KPIs

### ¿Por qué no hay CDF de entrada desde BDC?
BDC garantiza que el share siempre tiene el estado más reciente de S/4HANA.
La incrementalidad de entrada la gestiona BDC internamente — el receptor recibe un snapshot fresco.
Lo que sí necesitas es una estrategia para **acumular** esos nuevos datos en tu Gold existente.

### El watermark con `updated_at`
Agregamos una columna `updated_at` a `vbak_silver` para identificar qué filas son nuevas:
- Registros históricos → `updated_at = 2024-12-31` (fecha pasada)
- Registros nuevos → `updated_at = fecha del lote` (fecha posterior)
- El MERGE filtra: `WHERE updated_at = MAX(updated_at)` → solo procesa el lote más reciente

```
SIN MERGE strategy:
  Llegan 6 órdenes nuevas → borras gold_sales_kpis completo
  → Recalculas todo desde cero (N millones de filas)  ❌

CON MERGE strategy + updated_at watermark:
  Llegan 6 órdenes nuevas con updated_at = 2025-06-01
  → WHERE updated_at = MAX(updated_at) → filtra solo esas 6
  → Si el mes existe: UPDATE acumulando
  → Si el mes es nuevo: INSERT  ✅
```


In [0]:
# Agregar columna de control de actualización a vbak_silver
spark.sql('''
    ALTER TABLE vbak_silver
    ADD COLUMN updated_at TIMESTAMP
''')

# Setear fecha pasada para todos los registros históricos
spark.sql('''
    UPDATE vbak_silver
    SET updated_at = TIMESTAMP('2024-12-31 00:00:00')
    WHERE updated_at IS NULL
''')

print('OK: columna updated_at agregada a vbak_silver')
spark.sql('''
    SELECT updated_at, COUNT(*) AS filas
    FROM vbak_silver
    GROUP BY updated_at
    ORDER BY updated_at
''').show()


### Paso 1.1 — Estado actual del Gold antes de cualquier cambio

In [0]:
spark.sql('''
    SELECT year, month, sales_org, currency,
           num_orders,
           ROUND(total_revenue, 2)   AS total_revenue,
           ROUND(avg_order_value, 2) AS avg_order_value
    FROM gold_sales_kpis
    ORDER BY year DESC, month DESC, sales_org
    LIMIT 15
''').show(truncate=False)


### Paso 1.2 — Simular nuevas órdenes llegando del share BDC

En producción estas filas vendrían del share BDC:
`spark.read.table('sap_bdc_catalog.sales.SalesOrder')` filtradas por fecha.
Aquí las creamos manualmente con `updated_at = 2025-06-01` — posterior al histórico.

In [0]:
from pyspark.sql.functions import to_date, to_timestamp, col

# Lote 1 — Junio 2025 (mes nuevo — entrará por WHEN NOT MATCHED)
nuevas_ordenes = spark.createDataFrame([
    ('9000000001','20250601','0000001000','1000',15200000.0,'COP','TA','2025-06-01 08:00:00'),
    ('9000000002','20250605','0000001001','1000',18500000.0,'COP','TA','2025-06-01 08:00:00'),
    ('9000000003','20250610','0000001002','1000',22100000.0,'COP','TA','2025-06-01 08:00:00'),
    ('9000000004','20250603','0000002000','2000',   52000.0,'USD','TA','2025-06-01 08:00:00'),
    ('9000000005','20250612','0000002001','2000',   67500.0,'USD','TA','2025-06-01 08:00:00'),
    ('9000000006','20250601','0000003000','3000',31400000.0,'COP','TA','2025-06-01 08:00:00'),
], ['VBELN','ERDAT','KUNNR','VKORG','NETWR','WAERK','AUART','updated_at'])

nuevas_ordenes = (nuevas_ordenes
    .withColumn('ERDAT',      to_date(col('ERDAT'), 'yyyyMMdd'))
    .withColumn('updated_at', to_timestamp(col('updated_at')))
)

nuevas_ordenes.write.format('delta').mode('append').saveAsTable('vbak_silver')
print(f'OK: {nuevas_ordenes.count()} órdenes de junio 2025 insertadas en Silver')

# Verificar los dos grupos de fechas
spark.sql('''
    SELECT updated_at, COUNT(*) AS filas
    FROM vbak_silver
    GROUP BY updated_at
    ORDER BY updated_at
''').show()


### Paso 1.3 — MERGE Strategy con CASE WHEN al Gold

El MERGE lee directamente de `vbak_silver` filtrando por `updated_at = MAX(updated_at)`.
No se necesita temp view intermedia — Silver es la fuente de verdad.

- **WHEN MATCHED** → mes ya existe en Gold: acumula sumando
- **WHEN NOT MATCHED** → mes nuevo: inserta directamente
- **CASE WHEN** → recalcula el promedio ponderado correctamente

In [0]:
spark.sql('''
    MERGE INTO gold_sales_kpis AS g
    USING (
        SELECT
            YEAR(ERDAT)   AS year,
            MONTH(ERDAT)  AS month,
            VKORG         AS sales_org,
            WAERK         AS currency,
            COUNT(*)      AS num_orders_nuevas,
            SUM(NETWR)    AS revenue_nuevo,
            AVG(NETWR)    AS avg_nuevo
        FROM vbak_silver
        WHERE updated_at = (
            SELECT MAX(updated_at) FROM vbak_silver
        )
        GROUP BY 1, 2, 3, 4
    ) AS n
    ON  g.year      = n.year
    AND g.month     = n.month
    AND g.sales_org = n.sales_org
    AND g.currency  = n.currency

    WHEN MATCHED THEN UPDATE SET
        g.num_orders      = g.num_orders + n.num_orders_nuevas,
        g.total_revenue   = g.total_revenue + n.revenue_nuevo,
        g.avg_order_value = CASE
            WHEN (g.num_orders + n.num_orders_nuevas) > 0
            THEN (g.total_revenue + n.revenue_nuevo)
                 / (g.num_orders + n.num_orders_nuevas)
            ELSE 0
        END

    WHEN NOT MATCHED THEN INSERT (
        year, month, sales_org, currency,
        num_orders, total_revenue, avg_order_value
    ) VALUES (
        n.year, n.month, n.sales_org, n.currency,
        n.num_orders_nuevas, n.revenue_nuevo, n.avg_nuevo
    )
''')

print('OK: gold_sales_kpis actualizado — Lote 1 (WHEN NOT MATCHED)')
spark.sql('''
    SELECT year, month, sales_org, currency,
           num_orders,
           ROUND(total_revenue, 2)   AS total_revenue,
           ROUND(avg_order_value, 2) AS avg_order_value
    FROM gold_sales_kpis
    WHERE year = 2025
    ORDER BY month, sales_org
''').show(truncate=False)


### Paso 1.4 — Segundo lote: mismo mes, debe acumular (WHEN MATCHED)

Ahora llegan 2 órdenes más de junio 2025 con `updated_at = 2025-06-15`.
El MERGE debe detectar que junio ya existe y **acumular** — no insertar de nuevo.

In [0]:
from pyspark.sql.functions import to_date, to_timestamp, col

# Lote 2 — mismo mes junio 2025 (entrará por WHEN MATCHED — acumula)
nuevas_ordenes_2 = spark.createDataFrame([
    ('9000000007','20250620','0000001003','1000', 9800000.0,'COP','TA','2025-06-15 08:00:00'),
    ('9000000008','20250622','0000002002','2000',   38000.0,'USD','TA','2025-06-15 08:00:00'),
], ['VBELN','ERDAT','KUNNR','VKORG','NETWR','WAERK','AUART','updated_at'])

nuevas_ordenes_2 = (nuevas_ordenes_2
    .withColumn('ERDAT',      to_date(col('ERDAT'), 'yyyyMMdd'))
    .withColumn('updated_at', to_timestamp(col('updated_at')))
)

nuevas_ordenes_2.write.format('delta').mode('append').saveAsTable('vbak_silver')
print(f'OK: {nuevas_ordenes_2.count()} órdenes más de junio 2025 insertadas en Silver')

spark.sql('''
    SELECT updated_at, COUNT(*) AS filas
    FROM vbak_silver GROUP BY updated_at ORDER BY updated_at
''').show()


In [0]:
spark.sql('''
    MERGE INTO gold_sales_kpis AS g
    USING (
        SELECT
            YEAR(ERDAT)   AS year,
            MONTH(ERDAT)  AS month,
            VKORG         AS sales_org,
            WAERK         AS currency,
            COUNT(*)      AS num_orders_nuevas,
            SUM(NETWR)    AS revenue_nuevo,
            AVG(NETWR)    AS avg_nuevo
        FROM vbak_silver
        WHERE updated_at = (
            SELECT MAX(updated_at) FROM vbak_silver
        )
        GROUP BY 1, 2, 3, 4
    ) AS n
    ON  g.year      = n.year
    AND g.month     = n.month
    AND g.sales_org = n.sales_org
    AND g.currency  = n.currency

    WHEN MATCHED THEN UPDATE SET
        g.num_orders      = g.num_orders + n.num_orders_nuevas,
        g.total_revenue   = g.total_revenue + n.revenue_nuevo,
        g.avg_order_value = CASE
            WHEN (g.num_orders + n.num_orders_nuevas) > 0
            THEN (g.total_revenue + n.revenue_nuevo)
                 / (g.num_orders + n.num_orders_nuevas)
            ELSE 0
        END

    WHEN NOT MATCHED THEN INSERT (
        year, month, sales_org, currency,
        num_orders, total_revenue, avg_order_value
    ) VALUES (
        n.year, n.month, n.sales_org, n.currency,
        n.num_orders_nuevas, n.revenue_nuevo, n.avg_nuevo
    )
''')

print('OK: gold_sales_kpis actualizado — Lote 2 (WHEN MATCHED acumuló)')
spark.sql('''
    SELECT year, month, sales_org, currency,
           num_orders,
           ROUND(total_revenue, 2)   AS total_revenue,
           ROUND(avg_order_value, 2) AS avg_order_value
    FROM gold_sales_kpis
    WHERE year = 2025
    ORDER BY month, sales_org
''').show(truncate=False)


### ¿Por qué CASE WHEN en avg_order_value?

```
SIN CASE WHEN — INCORRECTO:
  Histórico:  10 órdenes, revenue = 100M  →  avg = 10M
  Lote nuevo:  3 órdenes, revenue =  45M  →  avg = 15M
  UPDATE avg = 15M  ← solo promedio de las 3 nuevas, pierde el histórico  ❌

CON CASE WHEN — CORRECTO:
  avg = (100M + 45M) / (10 + 3) = 11.15M  ← promedio ponderado completo  ✅
```

**Conexión con BDC:** Este MERGE es lo que haces después de recibir el snapshot
del share de BDC. El share tiene todas las órdenes actualizadas —
el `updated_at` watermark garantiza que solo procesas el lote nuevo.

### Paso 1.5 — Verificar desde el receptor (ejecutar en el trial)

Abre el trial (`info@ecotickets.co`) y ejecuta esta query.
Deberías ver junio 2025 con los datos acumulados — **sin haber tocado nada en el trial.**

```sql
SELECT
    year, month, sales_org, currency,
    num_orders,
    ROUND(total_revenue, 2)   AS total_revenue,
    ROUND(avg_order_value, 2) AS avg_order_value
FROM sap_gold_shared.sales.monthly_kpis
WHERE year = 2025
ORDER BY month, sales_org
```

> **El mensaje clave:** Delta Sharing apunta a los archivos Parquet del proveedor.
> Cuando el Gold del Free Edition cambió, el trial lo ve en la siguiente query.
> Nadie ejecutó nada en el receptor — eso es **zero-copy** en acción.

---
## Ejercicio 2 — CDF de Retorno: Gold → OpenSharing → Receptor

**Tabla:** `gold_customer_360` (distinta al Ejercicio 1 para no mezclar los flujos)

**Caso de negocio:** El modelo de churn actualizó los tiers de los mejores clientes.
Esos cambios deben llegar a SAP Datasphere — pero solo los clientes que cambiaron,
no los N clientes completos.

```
SIN CDF — receptor descarga todo:
  gold_customer_360 tiene 1,000 clientes
  Actualizas el tier de 50 clientes
  Datasphere descarga 1,000 filas otra vez  ❌

CON CDF — receptor descarga solo cambios:
  gold_customer_360 tiene 1,000 clientes
  Actualizas el tier de 50 clientes
  Datasphere descarga solo 50 filas  ✅
```

**Equivalente en BDC:** el `enableChangeDataFeed=true` del `Company_Clustering.ipynb`.
Cuando escribe `company_code_clusters`, Datasphere descarga solo los clusters
que cambiaron — no los 500 Company Codes completos.

### Paso 2.1 — Habilitar CDF en gold_customer_360

In [0]:
spark.sql('''
    ALTER TABLE gold_customer_360
    SET TBLPROPERTIES (
        'delta.enableChangeDataFeed'  = 'true',
        'delta.enableDeletionVectors' = 'false'
    )
''')
print('OK: CDF habilitado en gold_customer_360')
print('   enableChangeDataFeed  = true   <- solo viajan los cambios')
print('   enableDeletionVectors = false  <- compatibilidad con BDC/Datasphere')


### Paso 2.2 — Capturar versión actual antes del cambio

In [0]:
hist = spark.sql('DESCRIBE HISTORY gold_customer_360 LIMIT 1').collect()
version_antes = hist[0]['version']
print(f'Versión actual de gold_customer_360: {version_antes}')
print(f'El receptor leerá CDF desde versión: {version_antes + 1}')
print('(cualquier cambio posterior a esta versión será capturado por CDF)')


### Paso 2.3 — Simular actualización de tiers de clientes

En producción: el modelo de churn corrió en Databricks y clasificó
a los clientes top 10% por revenue como PLATINUM.
Este UPDATE simula el resultado de ese modelo.

In [0]:
spark.sql('''
    UPDATE gold_customer_360
    SET customer_tier = 'PLATINUM'
    WHERE total_revenue > (
        SELECT PERCENTILE(total_revenue, 0.90)
        FROM gold_customer_360
    )
''')

total_platinum = spark.sql(
    "SELECT COUNT(*) FROM gold_customer_360 WHERE customer_tier = 'PLATINUM'"
).collect()[0][0]

print(f'OK: {total_platinum} clientes actualizados a PLATINUM (top 10% revenue)')
print('Solo estos clientes viajarán al receptor via CDF — no toda la tabla')


### Paso 2.4 — Leer SOLO los cambios via CDF

In [0]:
cambios_cdf = (spark.read.format('delta')
    .option('readChangeFeed', 'true')
    .option('startingVersion', version_antes + 1)
    .table('gold_customer_360'))

total    = spark.table('gold_customer_360').count()
cambios  = cambios_cdf.count()

print(f'Total clientes en gold_customer_360:     {total:,}')
print(f'Registros que viajan al receptor (CDF):  {cambios}')
print(f'Ahorro: {total - cambios:,} filas que NO se descargan')


### Paso 2.5 — Clasificar tipos de cambio CDF

CDF genera hasta 4 tipos de registros por operación:

| Tipo | Descripción | ¿El receptor lo necesita? |
|------|-------------|---------------------------|
| `insert` | Fila nueva | ✅ Sí |
| `update_postimage` | Cómo quedó la fila DESPUÉS del UPDATE | ✅ Sí |
| `update_preimage` | Cómo era la fila ANTES del UPDATE | ❌ No |
| `delete` | Fila eliminada | ✅ Sí (marcar como borrado) |

In [0]:
# Recrear el DataFrame en la misma celda para evitar scope issues
cambios_cdf = (spark.read.format('delta')
    .option('readChangeFeed', 'true')
    .option('startingVersion', version_antes + 1)
    .table('gold_customer_360'))

# Registrar como vista SQL — más estable para columnas de sistema (_change_type)
cambios_cdf.createOrReplaceTempView('cambios_cdf_view')

total = spark.table('gold_customer_360').count()

print('Tipos de cambios detectados por CDF:')
spark.sql('''
    SELECT _change_type, COUNT(*) AS filas
    FROM cambios_cdf_view
    GROUP BY _change_type
''').show()

print('Lo que recibe SAP Datasphere (insert + update_postimage):')
spark.sql('''
    SELECT customer_id, customer_name,
           customer_tier, ROUND(total_revenue, 2) AS total_revenue,
           _change_type
    FROM cambios_cdf_view
    WHERE _change_type IN ('insert', 'update_postimage')
    ORDER BY total_revenue DESC
    LIMIT 10
''').show(truncate=False)

total_receptor = spark.sql('''
    SELECT COUNT(*) FROM cambios_cdf_view
    WHERE _change_type IN ('insert', 'update_postimage')
''').collect()[0][0]

print(f'Resumen:')
print(f'  Total clientes en la tabla:          {total:,}')
print(f'  Registros que Datasphere descarga:   {total_receptor}')
print(f'  Ahorro real:                         {total - total_receptor:,} filas')


### Paso 2.6 — Verificar desde el receptor (ejecutar en el trial)

Abre el trial y ejecuta esta query para confirmar que los clientes PLATINUM
ya aparecen — **sin haber ejecutado nada en el trial.**

```sql
SELECT
    customer_id,
    customer_name,
    customer_tier,
    ROUND(total_revenue, 2) AS total_revenue
FROM sap_gold_shared.customers.customer_360
WHERE customer_tier = 'PLATINUM'
ORDER BY total_revenue DESC
```

> **El mensaje clave:** El trial nunca ejecutó nada. El CDF registró el cambio
> en el Delta Log del proveedor. Delta Sharing propagó solo esas filas al receptor.
> Datasphere hace exactamente esto con `company_code_clusters` — descarga solo
> los clusters que cambiaron, no los 500 Company Codes completos.

### ¿Por qué `enableDeletionVectors = false`?

```
Delta Lake moderno usa Deletion Vectors para marcar borrados
(archivos .dv en el storage — muy eficientes para Databricks)

SAP Datasphere NO puede leer Deletion Vectors
→ enableDeletionVectors = false
→ Los borrados se representan como filas 'delete' en el CDF log
→ Datasphere SÍ puede procesarlas

Trade-off:
  false = escrituras ligeramente más lentas, compatible con BDC  ✅
  true  = más rápido pero incompatible con BDC/Datasphere  ❌

Regla: si la tabla va a ser compartida con BDC → siempre false
```

---
## Comparación final — Los dos patrones en Grupo Argos

| | Ejercicio 1 — MERGE Strategy | Ejercicio 2 — CDF Retorno |
|--|-------------------------------|---------------------------|
| **Tabla** | `vbak_silver` → `gold_sales_kpis` | `gold_customer_360` → OpenSharing |
| **Dirección** | SAP BDC → Databricks Gold | Databricks Gold → SAP Datasphere |
| **¿CDF de entrada?** | ❌ No — BDC envía snapshot | ✅ Sí — CDF=true en la tabla |
| **¿Por qué?** | El share ya está actualizado | Datasphere descarga solo cambios |
| **Patrón** | MERGE + CASE WHEN + `updated_at` | `readChangeFeed=true` |
| **En BDC** | Leer share → MERGE en Gold propio | `company_code_clusters` → Datasphere |
| **Verificar** | trial: `sap_gold_shared.sales.monthly_kpis` | trial: `sap_gold_shared.customers.customer_360` |

```
La autopista bidireccional SAP:

S/4HANA -CDC-> BDC Foundation -Delta Sharing-> SAP Databricks
                               (snapshot fresco)       |
                                         updated_at watermark
                                          MERGE Strategy
                                                 |
                                          Gold Tables
                                          (CDF = true)
                                                 |
                         SAP Datasphere <-Delta Sharing-+
                         (solo cambios via CDF)

IDA:    snapshot completo — la incrementalidad la gestiona BDC internamente
VUELTA: solo cambios — CDF lleva al receptor solo lo que modificó
```

---
## Preguntas de reflexión

1. **¿Por qué `CASE WHEN` en `avg_order_value` y no simplemente `avg = n.avg_nuevo`?**

2. **Si BDC no habilita CDF hacia SAP Databricks, ¿cómo sabe el notebook de clustering
   qué Company Codes son nuevos desde la última ejecución?**
   *(Pista: no lo sabe — siempre recalcula todo. ¿Cuándo es aceptable?)*

3. **`enableDeletionVectors=false` sacrifica rendimiento para ganar compatibilidad.
   ¿En qué escenario ese trade-off NO valdría la pena?**

4. **En el proyecto integrador tu tabla Gold tiene CDF=true. Si el compañero
   hace `SELECT` sin filtro de versión, ¿recibe todos los datos o solo los cambios?**
   *(Pista: `SELECT` normal siempre lee el estado actual — no el CDF log)*

5. **¿Por qué usamos `gold_customer_360` en el Ejercicio 2 y no `gold_sales_kpis`
   que ya teníamos del Ejercicio 1?**
   *(Pista: separación de responsabilidades — cada tabla tiene su propio flujo)*